# Shravana GPU Worker — Colab Setup

Run cells 1–4 in order. At the end you will get `REMOTE_GPU_URL` and
`REMOTE_GPU_TOKEN` values to paste into your local `.env` file.

**Prerequisites:** GPU runtime enabled (`Runtime → Change runtime type → T4 GPU`).

> **After an inactivity disconnect:** skip to **Cell 5 (Reconnect)** —
> models are still on disk, no need to re-download.

In [ ]:
# ── Cell 1: Clone repo & install dependencies ─────────────────────────────────
import os

# ❶ Set your GitHub repo URL here if the repo is private
REPO_URL = "https://github.com/bakamono12/Shravana.git"  # change if needed
REPO_DIR = "/content/Shravana"

# ❷ Optional: HuggingFace token for gated models (leave blank if not needed)
HF_TOKEN = ""  # e.g. "hf_xxxxxxxxxxxx"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)

# Install GPU torch first (Colab already has it, but pin for reproducibility)
!pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121

# Install the backend package (exposes app.ml.* to worker/server.py)
!pip install -q -e backend/

# Worker-specific extras
!pip install -q -r worker/requirements.txt

if HF_TOKEN:
    !huggingface-cli login --token {HF_TOKEN}

print("✓ Installation complete")

In [ ]:
# ── Cell 2: Download models ───────────────────────────────────────────────────
# Edit this list to match the models you want to serve from Colab.
# Available model names: qwen_lid, qwen3_asr, whisper_turbo, seamless_v2, qwen2_5_vl
MODELS_TO_DOWNLOAD = [
    "qwen_lid",
    "qwen3_asr",
    "whisper_turbo",
    "seamless_v2",
    "qwen2_5_vl",
]

import sys
sys.path.insert(0, "/content/Shravana")
sys.path.insert(0, "/content/Shravana/backend")

from huggingface_hub import snapshot_download
from pathlib import Path
from app.ml.registry import MODEL_CONFIGS

MODELS_DIR = Path("/content/Shravana/storage/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

for name in MODELS_TO_DOWNLOAD:
    if name == "whisper_turbo":
        print(f"⏭  {name}: faster-whisper downloads on first inference — skipping")
        continue
    cfg = MODEL_CONFIGS.get(name)
    if not cfg:
        print(f"⚠  Unknown model: {name}")
        continue
    dest = MODELS_DIR / name
    if dest.exists() and any(dest.rglob("*.safetensors")):
        print(f"✓  {name}: already downloaded")
        continue
    print(f"⬇  Downloading {name} ({cfg['repo_id']}) …")
    snapshot_download(repo_id=cfg["repo_id"], local_dir=str(dest))
    print(f"✓  {name}: done")

print("\n✓ All models ready")

In [ ]:
# ── Cell 3: Start worker server + Cloudflare tunnel ──────────────────────────
import os, subprocess, threading, time, secrets, re

os.chdir("/content/Shravana")

WORKER_TOKEN = secrets.token_hex(16)
WORKER_PORT = 8001

# ── Keep-alive: prevent Colab idle-timeout killing the runtime ────────────────
# Colab free tier disconnects after ~90 min with no computation.
# This daemon thread runs light work every 30 s to keep the kernel alive.
_keep_alive_running = True

def _keep_alive_loop():
    while _keep_alive_running:
        time.sleep(30)
        _ = sum(i * i for i in range(500))

_ka_thread = threading.Thread(target=_keep_alive_loop, daemon=True)
_ka_thread.start()
print("Keep-alive thread started (every 30 s)")

# ── Download cloudflared if not present ───────────────────────────────────────
CF_BIN = "/content/cloudflared"
if not os.path.exists(CF_BIN):
    print("Downloading cloudflared …")
    subprocess.run([
        "wget", "-q", "-O", CF_BIN,
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    ], check=True)
    os.chmod(CF_BIN, 0o755)

# ── Start uvicorn in background ───────────────────────────────────────────────
env = os.environ.copy()
env["WORKER_TOKEN"] = WORKER_TOKEN
env["WORKER_MODELS_DIR"] = "/content/Shravana/storage/models"
env["PYTHONPATH"] = "/content/Shravana:/content/Shravana/backend"
env["WORKER_API_VERSION"] = "2"  # signals two-stage pipeline support to backend

server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "worker.server:app",
     "--host", "0.0.0.0", "--port", str(WORKER_PORT), "--log-level", "info"],
    env=env,
)
print(f"Worker PID: {server_proc.pid}")
time.sleep(4)  # let uvicorn start

# ── Start Cloudflare tunnel and capture the public URL ────────────────────────
cf_log_path = "/tmp/cf_tunnel.log"
cf_proc = subprocess.Popen(
    [CF_BIN, "tunnel", "--url", f"http://localhost:{WORKER_PORT}"],
    stdout=open(cf_log_path, "w"), stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    time.sleep(2)
    log = open(cf_log_path).read()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    print("⚠  Could not parse tunnel URL — check /tmp/cf_tunnel.log")
else:
    print(f"\n✓ Tunnel ready: {public_url}")
    print("Run Cell 4 to get the .env snippet.")

In [ ]:
# ── Cell 4: Print .env snippet (copy-paste this into your local .env) ────────
if public_url:
    print("=" * 60)
    print("Add the following lines to your local backend .env file:")
    print("=" * 60)
    print(f"REMOTE_GPU_URL={public_url}")
    print(f"REMOTE_GPU_TOKEN={WORKER_TOKEN}")
    print("REMOTE_GPU_MODELS=ALL")
    print("=" * 60)
    print()
    print("Then restart the local backend (uvicorn) to pick up the new settings.")
    print("The tunnel URL changes each Colab session — run Cell 5 after reconnecting.")
else:
    print("public_url not set — re-run Cell 3 or check /tmp/cf_tunnel.log for errors.")

---
## Cell 5 — Reconnect after inactivity

Run **only this cell** when Colab wakes back up after going idle.
Models are still on disk — no need to repeat Cells 1–2.
It will print a fresh `.env` snippet with the new tunnel URL.

In [ ]:
# ── Cell 5: Reconnect after Colab inactivity disconnect ──────────────────────
import os, subprocess, threading, time, secrets, re

os.chdir("/content/Shravana")

WORKER_PORT = 8001
CF_BIN = "/content/cloudflared"

# Reuse the existing token if the kernel was not reset, otherwise generate new
try:
    WORKER_TOKEN  # defined in Cell 3 if kernel survived
except NameError:
    WORKER_TOKEN = secrets.token_hex(16)
    print("Generated new WORKER_TOKEN (kernel was reset)")

# Kill any stale server / tunnel processes
for proc_name in ["uvicorn", "cloudflared"]:
    subprocess.run(["pkill", "-f", proc_name], capture_output=True)
time.sleep(2)

# Keep-alive thread
def _keep_alive_loop():
    while True:
        time.sleep(30)
        _ = sum(i * i for i in range(500))

threading.Thread(target=_keep_alive_loop, daemon=True).start()
print("Keep-alive thread started")

# Restart uvicorn
env = os.environ.copy()
env["WORKER_TOKEN"] = WORKER_TOKEN
env["WORKER_MODELS_DIR"] = "/content/Shravana/storage/models"
env["PYTHONPATH"] = "/content/Shravana:/content/Shravana/backend"
env["WORKER_API_VERSION"] = "2"  # signals two-stage pipeline support to backend

server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "worker.server:app",
     "--host", "0.0.0.0", "--port", str(WORKER_PORT), "--log-level", "info"],
    env=env,
)
print(f"Worker PID: {server_proc.pid}")
time.sleep(4)

# Restart Cloudflare tunnel
cf_log_path = "/tmp/cf_tunnel.log"
cf_proc = subprocess.Popen(
    [CF_BIN, "tunnel", "--url", f"http://localhost:{WORKER_PORT}"],
    stdout=open(cf_log_path, "w"), stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    time.sleep(2)
    log = open(cf_log_path).read()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    print("⚠  Could not get tunnel URL — check /tmp/cf_tunnel.log")
else:
    print()
    print("=" * 60)
    print("Update your local backend .env with the NEW URL:")
    print("=" * 60)
    print(f"REMOTE_GPU_URL={public_url}")
    print(f"REMOTE_GPU_TOKEN={WORKER_TOKEN}")
    print("REMOTE_GPU_MODELS=ALL")
    print("=" * 60)
    print()
    print("Then restart the local backend to apply.")